<a href="https://colab.research.google.com/github/Saniyayes/AI-Projects/blob/main/Fine_Tuning_BERT_and_GPT.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Fine-Tuning Pretrained Models: BERT & GPT

This notebook demonstrates fine-tuning BERT for text classification and GPT for text generation.

## Install Required Libraries

In [ ]:
!pip install transformers torch

## Part A: Fine-Tuning BERT for Text Classification

In [ ]:

import torch
from transformers import BertTokenizer, BertForSequenceClassification

In [ ]:

texts = [
    "This course is amazing",
    "The session was boring",
    "Very informative training",
    "Worst explanation ever"
]
labels = [1, 0, 1, 0]


In [ ]:

tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")

encodings = tokenizer(
    texts,
    truncation=True,
    padding=True,
    max_length=128,
    return_tensors="pt"
)
labels_tensor = torch.tensor(labels)
labels_tensor

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tensor([1, 0, 1, 0])

In [ ]:

model = BertForSequenceClassification.from_pretrained(
    "bert-base-uncased",
    num_labels=2
)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-5)


Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [ ]:

model.train()
for epoch in range(10):
    optimizer.zero_grad()
    outputs = model(
        input_ids=encodings["input_ids"],
        attention_mask=encodings["attention_mask"],
        labels=labels_tensor
    )
    loss = outputs.loss
    loss.backward()
    optimizer.step()
    print(f"Epoch {epoch+1}, Loss: {loss.item():.4f}")


Epoch 1, Loss: 0.6780
Epoch 2, Loss: 0.6447
Epoch 3, Loss: 0.7008
Epoch 4, Loss: 0.4871
Epoch 5, Loss: 0.4955
Epoch 6, Loss: 0.5166
Epoch 7, Loss: 0.4042
Epoch 8, Loss: 0.4585
Epoch 9, Loss: 0.4377
Epoch 10, Loss: 0.4186


In [ ]:
import numpy as np
texts = [
    "This course is amazing",
    "The session was boring",
    "Very informative training",
    "Worst explanation ever"
]

model.eval()
test_text = "The session was boring"
inputs = tokenizer(test_text, return_tensors="pt")

with torch.no_grad():
    logits = model(**inputs).logits

prediction = (np.array(logits)[0])
prediction = np.argmax(prediction)
print(prediction)
print("Positive" if prediction.item() == 1 else "Negative")

0
Negative


## Part B: Fine-Tuning GPT for Text Generation

In [ ]:

from transformers import GPT2Tokenizer, GPT2LMHeadModel


In [ ]:

tokenizer = GPT2Tokenizer.from_pretrained("gpt2")
model = GPT2LMHeadModel.from_pretrained("gpt2")
tokenizer.pad_token = tokenizer.eos_token


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/548M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: gpt2
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...11}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

In [ ]:

train_texts = [
    "Deep learning is transforming education.",
    "Fine-tuning improves model performance.",
    "GPT models generate human-like text."
]

encodings = tokenizer(
    train_texts,
    return_tensors="pt",
    padding=True,
    truncation=True
)


In [ ]:

optimizer = torch.optim.AdamW(model.parameters(), lr=5e-5)

model.train()
for epoch in range(10):
    optimizer.zero_grad()
    outputs = model(
        input_ids=encodings["input_ids"],
        labels=encodings["input_ids"]
    )
    loss = outputs.loss
    loss.backward()
    optimizer.step()
    print(f"Epoch {epoch+1}, Loss: {loss.item():.4f}")


`loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


Epoch 1, Loss: 6.4655
Epoch 2, Loss: 5.2660
Epoch 3, Loss: 4.0139
Epoch 4, Loss: 3.6124
Epoch 5, Loss: 2.7251
Epoch 6, Loss: 2.5519
Epoch 7, Loss: 2.0696
Epoch 8, Loss: 1.1054
Epoch 9, Loss: 0.9931
Epoch 10, Loss: 1.2484


In [ ]:

model.eval()
prompt = ""
inputs = tokenizer(prompt, return_tensors="pt")

output = model.generate(
    **inputs,
    max_length=50,
    do_sample=True,
    top_k=50
)

print(tokenizer.decode(output[0], skip_special_tokens=True))

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


GPT models can perform real-time translation to many other languages.


In [ ]:
"Fine-tuning of llm is trending topic now a days"
"Fine-tuning of llm models improves model performance"